In [57]:
import os
import re
import requests
import nltk
import chromadb
from pypdf import PdfReader
from groq import Groq

nltk.download("punkt")
nltk.download("punkt_tab")

# ---- PASTE YOUR KEYS HERE (or set as environment variables) ----
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')   # only needed if EMBEDDING_PROVIDER = "openai"

HF_TOKEN = os.environ["HF_TOKEN"]
GROQ_API_KEY = os.environ["GROQ_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

PDF_PATH = "./policy.pdf"          # <-- put your PDF here
CHROMA_DIR = "./chromadb"         # local persistent folder for ChromaDB
COLLECTION_NAME = "health_policy"

# ---- EMBEDDING PROVIDER SWITCH ----
EMBEDDING_PROVIDER = "huggingface"   # "huggingface" (free) or "openai" (paid)
# EMBEDDING_PROVIDER = "openai"   # "huggingface" (free) or "openai" (paid)

HF_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
HF_EMBED_URL = f"https://router.huggingface.co/hf-inference/models/{HF_EMBED_MODEL}/pipeline/feature-extraction"

OPENAI_EMBED_MODEL = "text-embedding-3-small"
OPENAI_EMBED_URL = "https://api.openai.com/v1/embeddings"

GEN_MODEL = "llama-3.3-70b-versatile"

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [58]:
#====================================================================================================================================================================
#============================================================== READ THE PDF ========================================================================================
#====================================================================================================================================================================

def looks_like_heading(line):
    line = line.strip()
    if not line or len(line) > 60:
        return False
    if line.isupper():
        return True
    words = line.split()
    if len(words) >= 2 and sum(1 for w in words if w[:1].isupper()) / len(words) > 0.7:
        if not line.endswith("."):
            return True
    return False


def read_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    pages_data = []
    current_section = "General"

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        lines = text.split("\n")

        page_section = current_section
        page_lines = []
        for line in lines:
            if looks_like_heading(line):
                current_section = line.strip()
                page_section = current_section
            page_lines.append(line)

        pages_data.append({
            "page": page_num,
            "section": page_section,
            "text": "\n".join(page_lines).strip()
        })

    return pages_data


pages_data = read_pdf(PDF_PATH)
print(f"Read {len(pages_data)} pages from {PDF_PATH}")
print(pages_data[0] if pages_data else "No pages found")

Read 69 pages from ./policy.pdf
{'page': 1, 'section': 'D.1 Standard General Terms & Clauses  34', 'text': 'HDFC ERGO General Insurance Company Limited  \n \nPolicy Wording \nmy: Optima Secure  \n \nHDFC ERGO General Insurance Company Limited. IRDAI Reg. No.146 CIN: U66030MH2007PLC177117. Registered & Corporate Office: 6th Floor, \nLeela Business Park, Andheri-Kurla Road, Andheri (East), Mumbai – 400 059.  \nUIN: my: Optima Secure - HDFHLIP26058V082526 \n \n1 \nTable of Contents \nSr. No. Particulars Page No. \nPreamble 2 \nOperative Clause 2 \nA.1.1 Standard Definitions 2 \nA.1.2 Specific Definitions 2 \nB.1 Base Coverages 11 \nB.2 Optional Coverages 14 \nB.3 Renewal Benefit 27 \nC.1 Waiting Periods 30 \nC.2 Standard Exclusions 31 \nC.3 Specific Exclusions 33 \nD.1 Standard General Terms & Clauses  34 \nE Other Terms & Clauses 48 \nAnnexure A 52 \nAnnexure B 56 \nAnnexure C 60'}


In [59]:
#====================================================================================================================================================================
#============================================================== Semantic Chunking using NTLK ========================================================================
#====================================================================================================================================================================

from nltk.tokenize import sent_tokenize

CHUNK_TARGET_CHARS = 500   # rough target size per chunk
OVERLAP_SENTENCES = 2      # how many trailing sentences to carry into the next chunk

def chunk_pages(pages_data, target_chars=CHUNK_TARGET_CHARS, overlap_sentences=OVERLAP_SENTENCES):
    chunks = []

    for page_info in pages_data:
        text = page_info["text"]
        if not text:
            continue

        sentences = sent_tokenize(text)

        current_chunk = []
        current_len = 0

        for sentence in sentences:
            current_chunk.append(sentence)
            current_len += len(sentence)

            if current_len >= target_chars:
                chunks.append({
                    "text": " ".join(current_chunk),
                    "page": page_info["page"],
                    "section": page_info["section"]
                })
                # start the next chunk with the last few sentences of this one (overlap)
                current_chunk = current_chunk[-overlap_sentences:] if overlap_sentences else []
                current_len = sum(len(s) for s in current_chunk)

        # leftover sentences on this page (only keep if it has content beyond the carried-over overlap)
        if current_chunk and len(current_chunk) > overlap_sentences:
            chunks.append({
                "text": " ".join(current_chunk),
                "page": page_info["page"],
                "section": page_info["section"]
            })

    return chunks


chunks = chunk_pages(pages_data)
print(f"Created {len(chunks)} chunks")
print(chunks[0] if chunks else "No chunks created")

Created 474 chunks
{'text': 'HDFC ERGO General Insurance Company Limited  \n \nPolicy Wording \nmy: Optima Secure  \n \nHDFC ERGO General Insurance Company Limited. IRDAI Reg. No.146 CIN: U66030MH2007PLC177117. Registered & Corporate Office: 6th Floor, \nLeela Business Park, Andheri-Kurla Road, Andheri (East), Mumbai – 400 059. UIN: my: Optima Secure - HDFHLIP26058V082526 \n \n1 \nTable of Contents \nSr. No. Particulars Page No. Preamble 2 \nOperative Clause 2 \nA.1.1 Standard Definitions 2 \nA.1.2 Specific Definitions 2 \nB.1 Base Coverages 11 \nB.2 Optional Coverages 14 \nB.3 Renewal Benefit 27 \nC.1 Waiting Periods 30 \nC.2 Standard Exclusions 31 \nC.3 Specific Exclusions 33 \nD.1 Standard General Terms & Clauses  34 \nE Other Terms & Clauses 48 \nAnnexure A 52 \nAnnexure B 56 \nAnnexure C 60', 'page': 1, 'section': 'D.1 Standard General Terms & Clauses  34'}


In [60]:
#====================================================================================================================================================================
#============================================================== Create Embeddings ===================================================================================
#====================================================================================================================================================================

def get_embedding_hf(text):
    response = requests.post(
        HF_EMBED_URL,
        headers={"Authorization": f"Bearer {HF_TOKEN}"},
        json={"inputs": text, "options": {"wait_for_model": True}},
        timeout=60
    )
    response.raise_for_status()
    result = response.json()

    if isinstance(result, dict) and "error" in result:
        raise RuntimeError(f"HF Inference API error: {result['error']}")

    vector = result
    if isinstance(vector[0], list) and isinstance(vector[0][0], list):
        import numpy as np
        vector = np.mean(np.array(vector[0]), axis=0).tolist()
    elif isinstance(vector[0], list):
        vector = vector[0] if isinstance(vector[0][0], float) else vector
    return vector


def get_embedding_openai(text):
    response = requests.post(
        OPENAI_EMBED_URL,
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
        json={"input": text, "model": OPENAI_EMBED_MODEL}
    )
    response.raise_for_status()
    result = response.json()
    return result["data"][0]["embedding"]


def get_embedding(text):
    if EMBEDDING_PROVIDER == "openai":
        return get_embedding_openai(text)
    elif EMBEDDING_PROVIDER == "huggingface":
        return get_embedding_hf(text)
    else:
        raise ValueError(f"Unknown EMBEDDING_PROVIDER: {EMBEDDING_PROVIDER}")


test_vec = get_embedding("This is a test sentence.")
print(f"Using provider: {EMBEDDING_PROVIDER}")
print(f"Embedding length: {len(test_vec)}")

Using provider: huggingface
Embedding length: 384


In [61]:
#====================================================================================================================================================================
#============================================================== Store chunks + embeddings + metadata in ChromaDB ====================================================
#====================================================================================================================================================================


# Create/Open persistent database
client = chromadb.PersistentClient(path=CHROMA_DIR)

# Check whether collection already exists
existing_collections = [c.name for c in client.list_collections()]

if COLLECTION_NAME in existing_collections:
    print(f"Using existing collection: {COLLECTION_NAME}")
    collection = client.get_collection(COLLECTION_NAME)
else:
    print(f"Creating new collection: {COLLECTION_NAME}")
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )
    ids, embeddings, documents, metadatas = [], [], [], []

    for i, chunk in enumerate(chunks):
        emb = get_embedding(chunk["text"])
        ids.append(f"chunk_{i}")
        embeddings.append(emb)
        documents.append(chunk["text"])
        metadatas.append({"page": chunk["page"], "section": chunk["section"]})

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas
    )

print(f"Stored {collection.count()} chunks in ChromaDB at '{CHROMA_DIR}'")

Using existing collection: health_policy
Stored 281 chunks in ChromaDB at './chromadb'


In [62]:
#====================================================================================================================================================================
#============================================================== Top-K Retrieval (Cosine Similarity Matrix) ==========================================================
#====================================================================================================================================================================


def retrieve_top_k(query, k=3):
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    retrieved = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        retrieved.append({
            "text": doc,
            "section": meta["section"],
            "page": meta["page"],
            "distance": dist
        })
    return retrieved


sample_results = retrieve_top_k("What is the waiting period?", k=3)
for r in sample_results:
    print(f"[{r['section']}, p.{r['page']}] (distance={r['distance']:.4f})")
    print(r["text"][:150], "...\n")

[SECTION C. WAITING PERIOD AND EXCLUSIONS, p.30] (distance=0.4667)
Waiting Periods 
All the Waiting Periods and exclusions listed below shall be applicable individually for each Insured 
Person and claims shall be ass ...

[Policy Wording, p.43] (distance=0.5399)
d. Proposer shall be informed about the proposed loading with premium, specific 
Waiting Period or permanent exclusion (if any) through a counter offe ...

[Reimbursement of Hospitalization, Day Care, p.50] (distance=0.5517)
c. Pre-authorization issued by the Company shall be valid for 15 days from the date of 
issuance (or expiry of the Policy, whichever is earlier). d. T ...



In [63]:
#====================================================================================================================================================================
#============================================================== Generation (Groq - Llama 3.3 70B versatile) =========================================================
#====================================================================================================================================================================

groq_client = Groq(api_key=GROQ_API_KEY)

PROMPT_TEMPLATE = """You are a health insurance policy assistant. Answer ONLY using the provided context. If the answer isn't in the context, say so — do not guess. Context: {retrieved_chunks_with_section_path_and_page} Question: {user_query} Answer, and cite the section and page for each claim, e.g. (Source: Waiting Periods, p.12)."""


def format_context(retrieved_chunks):
    formatted = []
    for r in retrieved_chunks:
        formatted.append(f"[Section: {r['section']}, Page: {r['page']}]\n{r['text']}")
    return "\n\n".join(formatted)


def generate_answer(user_query, k=3):
    retrieved_chunks = retrieve_top_k(user_query, k=k)
    context = format_context(retrieved_chunks)

    prompt = PROMPT_TEMPLATE.format(
        retrieved_chunks_with_section_path_and_page=context,
        user_query=user_query
    )

    response = groq_client.chat.completions.create(
        model=GEN_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content, retrieved_chunks

In [64]:
#====================================================================================================================================================================
#============================================================== Ask a Question ======================================================================================
#====================================================================================================================================================================
# question = "What is the sum insured for room rent limits?"
while True:
    question = input("Ask me anything about the health insurance policy. ").strip().lower()
    answer, used_chunks = generate_answer(question, k=3)

    # print("QUESTION:", question)
    print("\nANSWER:\n", answer)

    # Ask if they want to continue
    quit_choice = input("Do you want to continue? (yes/no): ").strip().lower()
    if quit_choice == "no":
        print("Goodbye!")
        break

Ask me anything about the health insurance policy. is dental covered?

ANSWER:
 Dental treatment is covered, but only for accidental hospitalization. (Source: Optima Secure, p. 62) 

Additionally, the policy defines Dental Treatment as a treatment related to teeth or structures supporting teeth, including examinations, fillings, crowns, extractions, and surgery. (Source: Policy Wording, p. 4)
Do you want to continue? (yes/no): no
Goodbye!
